# ЛР-5: временные ряды и аномалии

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "pyspark==4.2.0" pandas numpy plotly

import numpy as np
import pandas as pd
import plotly.express as px

from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab05_TimeSeriesAnomalyDetection")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# 1. Синтетическая телеметрия нескольких роботов.
rng = np.random.default_rng(42)
robots = ["robot-0", "robot-1", "robot-2"]
n_per_robot = 1800  # 30 минут при 1 Гц

frames = []
start = pd.Timestamp("2026-08-31T09:00:00Z")

for ridx, robot in enumerate(robots):
    t = np.arange(n_per_robot)
    temperature = 44 + ridx + 0.002 * t + rng.normal(0, 0.25, n_per_robot)
    vibration = 0.20 + 0.03 * np.sin(t / 15) + rng.normal(0, 0.015, n_per_robot)

    # Вставка аномалий.
    temperature[850:870] += 10
    vibration[1200:1225] += 0.75

    frames.append(pd.DataFrame({
        "robot_id": robot,
        "timestamp": start + pd.to_timedelta(t, unit="s"),
        "temperature": temperature,
        "vibration": vibration,
    }))

pdf = pd.concat(frames, ignore_index=True)
sdf = spark.createDataFrame(pdf)

# 2. Z-score температуры внутри каждого робота.
w_robot = Window.partitionBy("robot_id")
sdf = (
    sdf
    .withColumn("temp_mean", F.avg("temperature").over(w_robot))
    .withColumn("temp_std", F.stddev_pop("temperature").over(w_robot))
    .withColumn("temp_z", (F.col("temperature") - F.col("temp_mean")) / F.col("temp_std"))
)

# 3. Скользящий RMS вибрации за последние 20 измерений.
w_roll = (
    Window.partitionBy("robot_id")
    .orderBy(F.col("timestamp").cast("long"))
    .rowsBetween(-19, 0)
)

sdf = sdf.withColumn(
    "vibration_rms",
    F.sqrt(F.avg(F.pow("vibration", 2)).over(w_roll))
)

# 4. Аномалия по двум независимым признакам.
sdf = (
    sdf
    .withColumn(
        "is_anomaly",
        (F.abs("temp_z") >= 3.0) | (F.col("vibration_rms") >= 0.45)
    )
)

anomalies = (
    sdf.filter("is_anomaly")
    .select("robot_id", "timestamp", "temperature", "temp_z", "vibration", "vibration_rms")
    .orderBy("robot_id", "timestamp")
)

print("Detected anomalies:", anomalies.count())
anomalies.show(20, truncate=False)

# 5. Визуализация.
plot_df = (
    sdf.select("robot_id", "timestamp", "temperature", "vibration_rms", "is_anomaly")
    .toPandas()
)

fig = px.line(
    plot_df,
    x="timestamp",
    y="vibration_rms",
    color="robot_id",
    title="Rolling RMS вибрации"
)
fig.show()

anom_pdf = plot_df[plot_df["is_anomaly"]]
fig2 = px.scatter(
    anom_pdf,
    x="timestamp",
    y="temperature",
    color="robot_id",
    title="События, классифицированные как аномальные"
)
fig2.show()

# 6. Сводка.
summary = (
    sdf.groupBy("robot_id")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.col("is_anomaly").cast("int")).alias("anomalies")
    )
    .withColumn("anomaly_rate", F.col("anomalies") / F.col("total"))
)
summary.show()

assert anomalies.count() > 0


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
